# Arrakis VisDrone Training

This notebook is meant for Kaggle GPU runs.

It will:
- clone the latest `ludia8888/Arrakis-Project` repository
- install the repository requirements
- auto-detect the attached VisDrone input from `/kaggle/input`
- convert raw VisDrone to YOLO format when needed
- train `yolo26s.pt` for strict `person / vehicle` detection with the repository defaults

Repository defaults currently mean:
- `epochs=50`
- `imgsz=1280`
- `batch=4`
- `workers=4`
- `device=0`


In [ ]:
from pathlib import Path

input_root = Path('/kaggle/input')
print('Top-level Kaggle inputs:')
for path in sorted(input_root.iterdir()):
    print('-', path)


In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

repo_dir = Path('/kaggle/working/Arrakis-Project')
repo_url = 'https://github.com/ludia8888/Arrakis-Project.git'

if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run(['git', 'clone', repo_url, str(repo_dir)], check=True)
subprocess.run([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    'torch==2.7.1',
    'torchvision==0.22.1',
    'torchaudio==2.7.1',
    '--index-url',
    'https://download.pytorch.org/whl/cu126',
], check=True)

subprocess.run([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '--no-deps',
    'ultralytics==8.4.21',
], check=True)

import torch
import ultralytics

print('Repository ready at:', repo_dir)
print('torch version:', torch.__version__)
print('ultralytics version:', ultralytics.__version__)


In [ ]:
from pathlib import Path
import subprocess
import sys

repo_dir = Path('/kaggle/working/Arrakis-Project')
train_script = repo_dir / 'kaggle_train_visdrone_yolo26s.py'

cmd = [sys.executable, str(train_script)]
print('Running command:')
print(' '.join(cmd))

subprocess.run(cmd, check=True, cwd=repo_dir)


In [ ]:
from pathlib import Path
import shutil

run_dir = Path('/kaggle/working/runs/visdrone/yolo26s-person-vehicle-p100')
weights_dir = run_dir / 'weights'
best_pt = weights_dir / 'best.pt'
last_pt = weights_dir / 'last.pt'

print('Run directory:', run_dir)
print('Weights directory exists:', weights_dir.exists())

if best_pt.exists():
    shutil.copy2(best_pt, Path('/kaggle/working/yolo26s_person_vehicle_best.pt'))
    print('Copied best.pt -> /kaggle/working/yolo26s_person_vehicle_best.pt')
else:
    print('best.pt not found')

if last_pt.exists():
    shutil.copy2(last_pt, Path('/kaggle/working/yolo26s_person_vehicle_last.pt'))
    print('Copied last.pt -> /kaggle/working/yolo26s_person_vehicle_last.pt')
else:
    print('last.pt not found')
